# Baseline YOLO26n IndoTraffic — final research pipeline

Notebook ini adalah versi final baseline **YOLO26n FPN-PAN** untuk eksperimen skripsi. Logika training dan evaluasi mempertahankan pipeline yang sudah lolos smoke test end-to-end; versi ini hanya merapikan dokumentasi, menjadikan mode default `research`, dan menambahkan inspeksi arsitektur/profiling model sebelum training.

Prinsip eksekusi:
- satu Kaggle batch menjalankan satu seed agar recovery sederhana;
- `train_batch=16` dipertahankan sebagai konfigurasi penelitian;
- `eval_batch=1` dipisahkan dari batch training untuk mencegah OOM pada evaluasi confidence rendah;
- validation memilih confidence threshold, sedangkan test hanya dipakai setelah threshold dikunci;
- `best.pt` dan `last.pt` diamankan ke W&B sebelum evaluasi;
- hasil audit, evaluasi, efficiency benchmark, grafik, dan artefak final disimpan untuk reproduksibilitas;
- profiling arsitektur hanya bersifat diagnostik; angka latency penelitian tetap berasal dari benchmark terkontrol di bagian efisiensi.

**Default notebook ini adalah `research`, seed 42, dan `run_all_seeds=False`.** Untuk seed berikutnya, ubah hanya `active_seed` menjadi `123` lalu `2026`; jangan mengubah hyperparameter lain.


## A. Persiapan platform

Aktifkan GPU dan Internet pada Kaggle. Tambahkan `ROBOFLOW_API_KEY` dan `WANDB_API_KEY` pada Kaggle Secrets. Notebook tidak pernah mencetak nilai secret.


## B. Instalasi dependensi

Versi utama dipin agar run lebih reproducible. PyTorch bawaan Kaggle tidak diganti.


In [ ]:
%pip install -q "ultralytics==8.4.155" "wandb==0.28.1" "roboflow==1.4.0" pycocotools pyyaml pandas matplotlib pillow scipy thop

from importlib.metadata import version

REQUIRED_VERSIONS = {
    "ultralytics": "8.4.155",
    "wandb": "0.28.1",
    "roboflow": "1.4.0",
}

for package, expected in REQUIRED_VERSIONS.items():
    installed = version(package)
    print(f"{package:12s}: {installed} (expected {expected})")
    if installed != expected:
        raise RuntimeError(
            f"Versi {package} tidak sesuai. Restart kernel lalu Run All kembali."
        )


## C. Environment dan direktori kerja


In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from PIL import Image, ImageDraw

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
PLATFORM = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/yolo26n_indotraffic")
elif IN_COLAB:
    WORK_ROOT = Path("/content/yolo26n_indotraffic")
else:
    WORK_ROOT = Path("./yolo26n_indotraffic")

WORK_ROOT = WORK_ROOT.resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if GPU_AVAILABLE else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "Tidak tersedia"
GPU_VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if GPU_AVAILABLE
    else 0.0
)

ENV_INFO = {
    "platform": PLATFORM,
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "wandb": wandb.__version__,
    "cuda_available": GPU_AVAILABLE,
    "cuda": torch.version.cuda,
    "gpu": GPU_NAME,
    "gpu_vram_gb": round(GPU_VRAM_GB, 2),
}

print(json.dumps(ENV_INFO, indent=2))
print("Work root:", WORK_ROOT)

if not GPU_AVAILABLE:
    raise RuntimeError("CUDA GPU tidak tersedia. Aktifkan GPU sebelum melanjutkan.")


## D. Konfigurasi eksperimen

Hanya `mode` yang perlu diubah antara uji pipeline dan eksperimen penelitian:

- `smoke`: 1 epoch, 3% train, subset val/test deterministik, benchmark singkat.
- `research`: konfigurasi training penuh dan evaluasi penuh.

Batch evaluasi sengaja `1` karena run sebelumnya OOM ketika evaluasi memakai batch training `16` dengan `conf=0.001` dan `nms=False`.


In [ ]:
CFG = {
    # --------------------------------------------------------
    # Execution mode
    # --------------------------------------------------------
    "mode": "research",  # "smoke" untuk regression test; "research" untuk eksperimen

    # --------------------------------------------------------
    # Dataset / model
    # --------------------------------------------------------
    "model": "yolo26n.pt",
    "workspace": "ilham-rafiqin",
    "project": "poltrada-original-raw-2",
    "dataset_version": 1,
    "class_names": [
        "Bus",
        "Mobil Penumpang",
        "Pejalan Kaki",
        "Sepeda Motor",
        "Truck",
        "Unmotorized",
    ],

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "seeds": [42, 123, 2026],
    "active_seed": 42,
    "run_all_seeds": False,

    # --------------------------------------------------------
    # Training — research configuration
    # --------------------------------------------------------
    "imgsz": 640,
    "train_batch": 16,
    "epochs": 100,
    "patience": 25,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "amp": True,
    "workers": 4,
    "save_period": 1,
    "artifact_period": 5,
    "close_mosaic": 10,
    "flipud": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,

    # --------------------------------------------------------
    # Evaluation — intentionally separate from training batch
    # --------------------------------------------------------
    "eval_batch": 1,
    "prediction_conf": 0.001,
    "prediction_iou": 0.7,

    # --------------------------------------------------------
    # Smoke mode
    # --------------------------------------------------------
    "smoke_epochs": 1,
    "smoke_train_fraction": 0.03,
    "smoke_eval_images_per_split": 256,
    "smoke_benchmark_warmup": 5,
    "smoke_benchmark_runs": 20,

    # --------------------------------------------------------
    # Research benchmark
    # --------------------------------------------------------
    "benchmark_warmup": 50,
    "benchmark_runs": 200,

    # --------------------------------------------------------
    # W&B / recovery
    # --------------------------------------------------------
    "wandb_project": "skripsi-yolo26n-indotraffic",
    "wandb_entity": None,
    "resume_checkpoint": None,
    "restore_bundle_artifact": None,
}

if CFG["mode"] not in {"smoke", "research"}:
    raise ValueError("CFG['mode'] harus 'smoke' atau 'research'.")

IS_SMOKE = CFG["mode"] == "smoke"
SEEDS_TO_RUN = (
    CFG["seeds"]
    if (CFG["mode"] == "research" and CFG["run_all_seeds"])
    else [CFG["active_seed"]]
)

if CFG["active_seed"] not in CFG["seeds"]:
    raise ValueError("active_seed harus merupakan salah satu CFG['seeds'].")

DATA_ROOT = WORK_ROOT / "dataset"
RUNS_ROOT = WORK_ROOT / "runs"
REPORTS_ROOT = WORK_ROOT / "reports"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

print("Mode            :", CFG["mode"])
print("Seed aktif      :", SEEDS_TO_RUN)
print("Training batch  :", CFG["train_batch"])
print("Evaluation batch:", CFG["eval_batch"])


## E. Secrets dan W&B


In [ ]:
def get_secret(name):
    value = os.getenv(name)
    if value:
        return value

    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass

    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass

    return None


ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")
WANDB_API_KEY = get_secret("WANDB_API_KEY")

if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY tidak ditemukan.")

WANDB_ENABLED = bool(WANDB_API_KEY)

if WANDB_ENABLED:
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("W&B: aktif")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B: nonaktif; hasil tetap disimpan lokal.")


### Recovery bundle sebelumnya (opsional)

Bagian ini tidak diperlukan untuk smoke test pertama. Ia dipertahankan untuk resume/penggabungan seed di tahap penelitian.


In [ ]:
RESTORE_ROOT = WORK_ROOT / "restored_bundle"
if CFG["restore_bundle_artifact"]:
    if not WANDB_ENABLED: raise RuntimeError("Pemulihan W&B membutuhkan WANDB_API_KEY.")
    artifact = wandb.Api().artifact(CFG["restore_bundle_artifact"])
    download_dir = Path(artifact.download(root=str(RESTORE_ROOT / "download")))
    zip_candidates = list(download_dir.rglob("*.zip"))
    if not zip_candidates: raise FileNotFoundError("ZIP tidak ditemukan di artefak W&B.")
    unpack_dir = RESTORE_ROOT / "unpacked"; unpack_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(zip_candidates[0]), str(unpack_dir))
    print("Bundle lama dipulihkan ke:", unpack_dir)
else:
    print("Tidak ada bundle W&B yang dipulihkan.")


## F. Download dan siapkan dataset

Dataset diunduh dari versi Roboflow yang tetap. `data.yaml` dicari secara defensif agar tidak bergantung pada satu struktur folder SDK.


In [ ]:
# ============================================================
# F. DOWNLOAD & PREPARE DATASET
# ============================================================

from roboflow import Roboflow


# ------------------------------------------------------------
# 1. Cek apakah dataset sudah tersedia
# ------------------------------------------------------------

source_yaml_candidates = (
    list(DATA_ROOT.rglob("data.yaml"))
    if DATA_ROOT.exists()
    else []
)


if source_yaml_candidates:

    SOURCE_YAML = source_yaml_candidates[0]

    print(
        "[INFO] Menggunakan dataset yang sudah tersedia:"
    )

    print(
        SOURCE_YAML.parent
    )


else:

    # --------------------------------------------------------
    # 2. Bersihkan folder kosong / download gagal sebelumnya
    # --------------------------------------------------------

    if DATA_ROOT.exists():

        print(
            "[INFO] Folder dataset ada tetapi "
            "data.yaml tidak ditemukan."
        )

        print(
            "[INFO] Membersihkan folder dataset "
            "sebelum download ulang..."
        )

        shutil.rmtree(
            DATA_ROOT
        )


    # --------------------------------------------------------
    # 3. Download exact Roboflow version
    # --------------------------------------------------------

    print(
        "[INFO] Download dataset dari Roboflow..."
    )

    print(
        "Workspace:",
        CFG["workspace"],
    )

    print(
        "Project  :",
        CFG["project"],
    )

    print(
        "Version  :",
        CFG["dataset_version"],
    )


    rf = Roboflow(
        api_key=ROBOFLOW_API_KEY
    )

    rf_project = (
        rf
        .workspace(
            CFG["workspace"]
        )
        .project(
            CFG["project"]
        )
    )


    downloaded = (
        rf_project
        .version(
            CFG["dataset_version"]
        )
        .download(
            "yolov8",
            location=str(DATA_ROOT),
            overwrite=True,
        )
    )


    # --------------------------------------------------------
    # 4. Jangan mengasumsikan lokasi data.yaml
    # --------------------------------------------------------

    candidate_roots = [
        DATA_ROOT,
        Path(downloaded.location),
    ]


    source_yaml_candidates = []

    for root in candidate_roots:

        if root.exists():

            source_yaml_candidates.extend(
                root.rglob(
                    "data.yaml"
                )
            )


    # Hilangkan duplicate path
    source_yaml_candidates = list(
        dict.fromkeys(
            source_yaml_candidates
        )
    )


    if not source_yaml_candidates:

        print()
        print(
            "[DEBUG] Isi DATA_ROOT:"
        )

        if DATA_ROOT.exists():

            for path in DATA_ROOT.rglob("*"):

                print(
                    path.relative_to(
                        DATA_ROOT
                    )
                )


        raise FileNotFoundError(
            "Dataset selesai diproses Roboflow, "
            "tetapi data.yaml tidak ditemukan.\n"
            f"DATA_ROOT = {DATA_ROOT}\n"
            f"downloaded.location = "
            f"{downloaded.location}"
        )


    SOURCE_YAML = (
        source_yaml_candidates[0]
    )


# ============================================================
# 5. VALIDATE YAML
# ============================================================

print()
print("=" * 70)
print("DATASET SOURCE")
print("=" * 70)

print(
    "SOURCE_YAML:",
    SOURCE_YAML,
)


raw_cfg = yaml.safe_load(
    SOURCE_YAML.read_text(
        encoding="utf-8"
    )
)


names = raw_cfg.get(
    "names",
    []
)


if isinstance(
    names,
    dict,
):

    names = [
        names[k]
        for k in sorted(
            names,
            key=lambda x: int(x),
        )
    ]

else:

    names = list(
        names
    )


print(
    "Classes:",
    names,
)


if names != CFG["class_names"]:

    raise ValueError(
        "Urutan kelas dataset berbeda.\n"
        f"Expected : {CFG['class_names']}\n"
        f"Found    : {names}"
    )


# ============================================================
# 6. RESOLVE SPLIT PATH
# ============================================================

def resolve_split(
    key,
    aliases=(),
):

    value = raw_cfg.get(
        key
    )

    candidates = []


    # --------------------------------------------------------
    # Path berdasarkan data.yaml
    # --------------------------------------------------------

    if value:

        p = Path(
            str(value)
        )

        if p.is_absolute():

            candidates.append(
                p
            )

        else:

            candidates.append(
                (
                    SOURCE_YAML.parent
                    / p
                ).resolve()
            )


    # --------------------------------------------------------
    # Fallback struktur standar Roboflow
    # --------------------------------------------------------

    for alias in aliases:

        candidates.append(
            (
                SOURCE_YAML.parent
                / alias
                / "images"
            ).resolve()
        )


    # --------------------------------------------------------
    # Cari yang benar-benar tersedia
    # --------------------------------------------------------

    for candidate in candidates:

        if candidate.exists():

            return candidate


    raise FileNotFoundError(
        f"Split '{key}' tidak ditemukan.\n"
        f"Kandidat:\n"
        + "\n".join(
            str(path)
            for path in candidates
        )
    )


SPLIT_IMAGES = {

    "train": resolve_split(
        "train",
        (
            "train",
        ),
    ),

    "val": resolve_split(
        "val",
        (
            "valid",
            "val",
        ),
    ),

    "test": resolve_split(
        "test",
        (
            "test",
        ),
    ),
}


# ============================================================
# 7. CREATE RUNTIME YAML
# ============================================================

runtime_cfg = {

    "path": str(
        SOURCE_YAML.parent.resolve()
    ),

    "train": str(
        SPLIT_IMAGES["train"]
    ),

    "val": str(
        SPLIT_IMAGES["val"]
    ),

    "test": str(
        SPLIT_IMAGES["test"]
    ),

    "nc": len(
        names
    ),

    "names": names,
}


RUNTIME_YAML = (
    WORK_ROOT
    / "data_runtime.yaml"
)


RUNTIME_YAML.write_text(
    yaml.safe_dump(
        runtime_cfg,
        sort_keys=False,
        allow_unicode=True,
    ),
    encoding="utf-8",
)


# ============================================================
# 8. FINAL CHECK
# ============================================================

print()
print("=" * 70)
print("DATASET READY")
print("=" * 70)

print(
    "Source YAML :",
    SOURCE_YAML,
)

print(
    "Train       :",
    SPLIT_IMAGES["train"],
)

print(
    "Validation  :",
    SPLIT_IMAGES["val"],
)

print(
    "Test        :",
    SPLIT_IMAGES["test"],
)

print(
    "Runtime YAML:",
    RUNTIME_YAML,
)

print("=" * 70)


## G. Normalisasi label polygon → bounding box

Dataset penelitian adalah object detection. Label 5 kolom (`class xc yc w h`) dibiarkan apa adanya. Baris polygon YOLO dikonversi secara deterministik menjadi *enclosing bounding box*. File asli yang berubah dibackup dan conversion report disimpan.


In [ ]:
# ============================================================
# G. NORMALIZE LABEL FORMAT FOR OBJECT DETECTION
# ============================================================
#
# Tujuan:
# - YOLO bbox (5 kolom) -> dibiarkan
# - YOLO polygon        -> dikonversi menjadi bounding box
# - format lain         -> dihentikan
#
# Tidak mengubah split, class ID, atau image.
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


BACKUP_ROOT = (
    REPORTS_ROOT
    / "label_backup_before_polygon_conversion"
)

BACKUP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


conversion_rows = []


def polygon_to_bbox(parts):
    """
    Convert:
        class x1 y1 x2 y2 ... xn yn

    menjadi:
        class xc yc width height
    """

    class_id = parts[0]

    coords = list(
        map(
            float,
            parts[1:]
        )
    )

    # Polygon minimal 3 titik:
    # class + 6 coordinate = 7 columns
    if len(coords) < 6:
        raise ValueError(
            "Polygon memiliki kurang dari 3 titik."
        )

    if len(coords) % 2 != 0:
        raise ValueError(
            "Jumlah koordinat polygon bukan pasangan x,y."
        )


    xs = coords[0::2]
    ys = coords[1::2]


    # Pastikan normalized coordinates masuk akal
    if not all(
        0 <= value <= 1
        for value in xs + ys
    ):
        raise ValueError(
            "Koordinat polygon berada di luar rentang [0,1]."
        )


    x_min = min(xs)
    x_max = max(xs)

    y_min = min(ys)
    y_max = max(ys)


    width = x_max - x_min
    height = y_max - y_min


    if width <= 0 or height <= 0:
        raise ValueError(
            "Polygon menghasilkan bbox dengan ukuran <= 0."
        )


    x_center = (
        x_min + x_max
    ) / 2

    y_center = (
        y_min + y_max
    ) / 2


    return (
        f"{class_id} "
        f"{x_center:.10f} "
        f"{y_center:.10f} "
        f"{width:.10f} "
        f"{height:.10f}"
    )


# ============================================================
# Find labels that need normalization
# ============================================================

converted_files = 0
converted_objects = 0


for split, image_dir in SPLIT_IMAGES.items():

    image_dir = Path(
        image_dir
    )

    label_dir = (
        image_dir.parent
        / "labels"
    )


    if not label_dir.exists():
        continue


    for label_path in sorted(
        label_dir.rglob("*.txt")
    ):

        original_text = (
            label_path.read_text(
                encoding="utf-8"
            )
        )

        lines = (
            original_text
            .splitlines()
        )


        new_lines = []

        file_changed = False


        for line_number, raw_line in enumerate(
            lines,
            start=1,
        ):

            line = raw_line.strip()


            if not line:
                continue


            parts = line.split()


            # ------------------------------------------------
            # Standard YOLO detection
            # ------------------------------------------------

            if len(parts) == 5:

                new_lines.append(
                    line
                )

                continue


            # ------------------------------------------------
            # YOLO polygon:
            # class + >= 3 coordinate pairs
            # ------------------------------------------------

            if (
                len(parts) >= 7
                and len(parts) % 2 == 1
            ):

                bbox_line = polygon_to_bbox(
                    parts
                )

                new_lines.append(
                    bbox_line
                )

                file_changed = True
                converted_objects += 1


                conversion_rows.append({
                    "split": split,
                    "label": str(label_path),
                    "line": line_number,
                    "original_columns": len(parts),
                    "action": "polygon_to_bbox",
                    "original": line,
                    "converted": bbox_line,
                })

                continue


            # ------------------------------------------------
            # Unknown format
            # ------------------------------------------------

            raise RuntimeError(
                "Format label tidak dikenali:\n"
                f"File : {label_path}\n"
                f"Line : {line_number}\n"
                f"Cols : {len(parts)}\n"
                f"Raw  : {line}"
            )


        # ====================================================
        # Write only affected files
        # ====================================================

        if file_changed:

            relative_path = (
                label_path
                .relative_to(
                    label_dir
                )
            )


            backup_path = (
                BACKUP_ROOT
                / split
                / relative_path
            )


            backup_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )


            shutil.copy2(
                label_path,
                backup_path,
            )


            label_path.write_text(
                "\n".join(
                    new_lines
                ) + "\n",
                encoding="utf-8",
            )


            converted_files += 1


# ============================================================
# Conversion report
# ============================================================

CONVERSION_DF = pd.DataFrame(
    conversion_rows
)


CONVERSION_REPORT = (
    REPORTS_ROOT
    / "polygon_to_bbox_conversion.csv"
)


if not CONVERSION_DF.empty:

    CONVERSION_DF.to_csv(
        CONVERSION_REPORT,
        index=False,
    )


print("=" * 70)
print("LABEL NORMALIZATION COMPLETE")
print("=" * 70)

print(
    "Files converted  :",
    converted_files,
)

print(
    "Objects converted:",
    converted_objects,
)

print(
    "Backup directory :",
    BACKUP_ROOT,
)

print(
    "Conversion report:",
    CONVERSION_REPORT,
)


if not CONVERSION_DF.empty:

    display(
        CONVERSION_DF[
            [
                "split",
                "label",
                "line",
                "original_columns",
                "action",
            ]
        ]
    )


## H. Audit dataset

Audit memeriksa citra rusak, label hilang/kosong, format YOLO, class ID, koordinat, batas box, serta duplicate byte antarsplit. Overflow sangat kecil di tepi gambar diberi toleransi `1e-5` untuk menghindari false positive akibat floating-point.


In [ ]:
# ============================================================
# G. DATASET AUDIT
# ============================================================

from pathlib import Path
import hashlib

import pandas as pd
from PIL import Image


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# Toleransi floating-point untuk bbox yang menyentuh tepi citra.
# Dataset kita memiliki overflow maksimum sekitar 5e-6.
BOUNDARY_EPS = 1e-5

NON_BLOCKING_STATUS = {
    "ok",
    "empty_label",
}


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def label_dir_for(image_dir: Path) -> Path:
    """Dapatkan direktori label pasangan dari direktori images."""

    if image_dir.name == "images":
        return image_dir.parent / "labels"

    return Path(
        str(image_dir).replace(
            "images",
            "labels",
        )
    )


def sha256_file(path: Path) -> str:
    """Hitung SHA256 tanpa membaca seluruh file ke RAM sekaligus."""

    hasher = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


def validate_label_file(
    label_path: Path,
    num_classes: int,
    boundary_eps: float = BOUNDARY_EPS,
):
    """
    Validasi satu file label YOLO detection.

    Returns
    -------
    status : str
    object_count : int
    issue : dict | None
    """

    # --------------------------------------------------------
    # Label tidak tersedia
    # --------------------------------------------------------

    if not label_path.exists():
        return (
            "missing_label",
            0,
            {
                "issue": "missing_label",
                "line": None,
                "raw": None,
            },
        )


    # --------------------------------------------------------
    # Read label
    # --------------------------------------------------------

    lines = [
        line.strip()
        for line in label_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]


    if not lines:
        return (
            "empty_label",
            0,
            None,
        )


    object_count = 0


    # --------------------------------------------------------
    # Validate each YOLO annotation
    # --------------------------------------------------------

    for line_no, line in enumerate(
        lines,
        start=1,
    ):

        parts = line.split()


        # YOLO detection harus:
        # class xc yc width height
        if len(parts) != 5:

            return (
                f"invalid_label_line_{line_no}:"
                "jumlah kolom bukan 5",

                object_count,

                {
                    "issue": "column_count",
                    "line": line_no,
                    "n_columns": len(parts),
                    "raw": line,
                },
            )


        try:

            cls, xc, yc, bw, bh = map(
                float,
                parts,
            )

        except ValueError:

            return (
                f"invalid_label_line_{line_no}:"
                "nilai bukan numerik",

                object_count,

                {
                    "issue": "non_numeric",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Class ID
        # ----------------------------------------------------

        if (
            not cls.is_integer()
            or not 0 <= int(cls) < num_classes
        ):

            return (
                f"invalid_label_line_{line_no}:"
                "class id invalid",

                object_count,

                {
                    "issue": "invalid_class",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Basic normalized-coordinate validation
        # ----------------------------------------------------

        if not (
            0 <= xc <= 1
            and 0 <= yc <= 1
            and 0 < bw <= 1
            and 0 < bh <= 1
        ):

            return (
                f"invalid_label_line_{line_no}:"
                "koordinat invalid",

                object_count,

                {
                    "issue": "invalid_coordinate",
                    "line": line_no,
                    "raw": line,
                },
            )


        # ----------------------------------------------------
        # Bounding-box boundaries
        # ----------------------------------------------------

        x1 = xc - (bw / 2)
        y1 = yc - (bh / 2)

        x2 = xc + (bw / 2)
        y2 = yc + (bh / 2)


        overflow = max(
            0.0,
            -x1,
            -y1,
            x2 - 1.0,
            y2 - 1.0,
        )


        # Hanya dianggap invalid jika melebihi
        # toleransi floating-point.
        if overflow > boundary_eps:

            return (
                f"invalid_label_line_{line_no}:"
                "box keluar citra",

                object_count,

                {
                    "issue": "out_of_bounds",
                    "line": line_no,
                    "overflow": overflow,
                    "raw": line,
                },
            )


        object_count += 1


    return (
        "ok",
        object_count,
        None,
    )


# ============================================================
# Run audit
# ============================================================

audit_rows = []
issue_rows = []

seen_hashes = {}


for split, image_dir in SPLIT_IMAGES.items():

    image_dir = Path(image_dir)

    label_dir = label_dir_for(
        image_dir
    )


    image_paths = sorted(
        path
        for path in image_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTS
        )
    )


    print(
        f"[AUDIT] {split}: "
        f"{len(image_paths)} images"
    )


    for image_path in image_paths:

        relative_label = (
            image_path
            .relative_to(image_dir)
            .with_suffix(".txt")
        )

        label_path = (
            label_dir
            / relative_label
        )


        row = {
            "split": split,
            "image": str(image_path),
            "label": str(label_path),
            "objects": 0,
            "status": "ok",
            "width": None,
            "height": None,
        }


        # ----------------------------------------------------
        # Validate image
        # ----------------------------------------------------

        try:

            with Image.open(
                image_path
            ) as image:

                image.verify()


            with Image.open(
                image_path
            ) as image:

                row["width"], row["height"] = (
                    image.size
                )


        except Exception as exc:

            row["status"] = (
                f"corrupt_image:"
                f"{type(exc).__name__}"
            )


        # ----------------------------------------------------
        # Validate annotation
        # ----------------------------------------------------

        if row["status"] == "ok":

            (
                status,
                object_count,
                issue,
            ) = validate_label_file(
                label_path=label_path,
                num_classes=len(names),
            )


            row["status"] = status
            row["objects"] = object_count


            if issue is not None:

                issue_rows.append({
                    "split": split,
                    "image": str(image_path),
                    "label": str(label_path),
                    **issue,
                })


        # ----------------------------------------------------
        # Hashes
        # ----------------------------------------------------

        image_hash = sha256_file(
            image_path
        )


        label_hash = (
            sha256_file(label_path)
            if label_path.exists()
            else None
        )


        row["sha256"] = image_hash
        row["label_sha256"] = label_hash


        seen_hashes.setdefault(
            image_hash,
            [],
        ).append(
            (
                split,
                str(image_path),
            )
        )


        audit_rows.append(
            row
        )


# ============================================================
# Build audit tables
# ============================================================

AUDIT_DF = pd.DataFrame(
    audit_rows
)

ISSUES_DF = pd.DataFrame(
    issue_rows
)


# ------------------------------------------------------------
# Cross-split duplicate detection
# ------------------------------------------------------------

duplicate_leaks = {
    digest: occurrences

    for digest, occurrences
    in seen_hashes.items()

    if len({
        split
        for split, _
        in occurrences
    }) > 1
}


# ------------------------------------------------------------
# Dataset fingerprint
# ------------------------------------------------------------

fingerprint_payload = "\n".join(
    sorted(
        (
            f"{row['split']}|"
            f"{Path(row['image']).name}|"
            f"{row['sha256']}|"
            f"{row['label_sha256']}"
        )

        for row in audit_rows
    )
)


DATASET_FINGERPRINT = (
    hashlib
    .sha256(
        fingerprint_payload.encode(
            "utf-8"
        )
    )
    .hexdigest()
)


# ============================================================
# Save audit report
# ============================================================

AUDIT_DF.to_csv(
    REPORTS_ROOT
    / "dataset_audit.csv",
    index=False,
)


if not ISSUES_DF.empty:

    ISSUES_DF.to_csv(
        REPORTS_ROOT
        / "dataset_audit_issues.csv",
        index=False,
    )


# ============================================================
# Summary
# ============================================================

summary = (
    AUDIT_DF
    .groupby(
        [
            "split",
            "status",
        ],
        as_index=False,
    )
    .agg(
        images=("image", "count"),
        objects=("objects", "sum"),
    )
)


display(
    summary
)


print()
print(
    "Dataset fingerprint:",
    DATASET_FINGERPRINT,
)

print(
    "Duplicate cross-split:",
    len(duplicate_leaks),
)

print(
    "Empty labels:",
    int(
        (
            AUDIT_DF["status"]
            == "empty_label"
        ).sum()
    ),
)


# ============================================================
# Blocking issues
# ============================================================

blocking = AUDIT_DF[
    ~AUDIT_DF["status"].isin(
        NON_BLOCKING_STATUS
    )
].copy()


print()
print(
    "Blocking rows:",
    len(blocking),
)


if len(blocking):

    display(
        blocking[
            [
                "split",
                "image",
                "label",
                "status",
            ]
        ]
    )


    if not ISSUES_DF.empty:

        print()
        print(
            "Detail masalah anotasi:"
        )

        display(
            ISSUES_DF
        )


if duplicate_leaks:

    print()
    print(
        "Duplicate images across splits:"
    )

    for digest, occurrences in (
        duplicate_leaks.items()
    ):

        print(
            digest,
            occurrences,
        )


# ============================================================
# Final gate
# ============================================================

if (
    len(blocking) > 0
    or len(duplicate_leaks) > 0
):

    raise RuntimeError(
        "Audit belum lolos. "
        "Lihat blocking dan ISSUES_DF "
        "di atas sebelum training."
    )


print()
print("=" * 70)
print("DATASET AUDIT PASSED")
print("=" * 70)


### Sanity check visual annotation


In [ ]:
def read_yolo_boxes(label_path, width, height):
    boxes = []
    label_path = Path(label_path)

    if not label_path.exists():
        return boxes

    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        cls, xc, yc, bw, bh = map(float, line.split())
        boxes.append(
            (
                int(cls),
                (xc - bw / 2) * width,
                (yc - bh / 2) * height,
                (xc + bw / 2) * width,
                (yc + bh / 2) * height,
            )
        )

    return boxes


sample = (
    AUDIT_DF[AUDIT_DF.status == "ok"]
    .groupby("split", group_keys=False)
    .sample(n=2, random_state=CFG["active_seed"])
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for ax, row in zip(axes, sample.itertuples()):
    image = Image.open(row.image).convert("RGB")
    draw = ImageDraw.Draw(image)

    for cls, x1, y1, x2, y2 in read_yolo_boxes(
        row.label, row.width, row.height
    ):
        draw.rectangle((x1, y1, x2, y2), width=max(2, row.width // 500))
        draw.text((x1, y1), names[cls])

    ax.imshow(image)
    ax.set_title(f"{row.split}: {Path(row.image).name}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## I. Data aktif untuk smoke/research

Smoke mode memakai subset validation dan test yang dipilih secara deterministik. Ini hanya untuk menguji jalur kode end-to-end dan **bukan hasil penelitian**. Research mode otomatis memakai split penuh tanpa subset.


In [ ]:
def canonical_path(path):
    return str(Path(path).resolve())


ACTIVE_EVAL_PATHS = {"val": None, "test": None}
SMOKE_DATA_YAML = WORK_ROOT / "data_smoke.yaml"

if IS_SMOKE:
    smoke_cfg = dict(runtime_cfg)

    for split in ("val", "test"):
        split_df = AUDIT_DF[
            (AUDIT_DF["split"] == split)
            & (AUDIT_DF["status"].isin(NON_BLOCKING_STATUS))
        ].copy()

        n = min(CFG["smoke_eval_images_per_split"], len(split_df))
        selected = split_df.sample(
            n=n,
            random_state=CFG["active_seed"],
        ).sort_values("image")

        image_paths = [canonical_path(path) for path in selected["image"]]
        manifest_path = WORK_ROOT / f"smoke_{split}_images.txt"
        manifest_path.write_text("\n".join(image_paths) + "\n", encoding="utf-8")

        smoke_cfg[split] = str(manifest_path)
        ACTIVE_EVAL_PATHS[split] = set(image_paths)

    SMOKE_DATA_YAML.write_text(
        yaml.safe_dump(smoke_cfg, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )
    ACTIVE_DATA_YAML = SMOKE_DATA_YAML
else:
    ACTIVE_DATA_YAML = RUNTIME_YAML

print("Active data YAML:", ACTIVE_DATA_YAML)
if IS_SMOKE:
    print("Smoke val images :", len(ACTIVE_EVAL_PATHS["val"]))
    print("Smoke test images:", len(ACTIVE_EVAL_PATHS["test"]))


## J. Training, checkpoint, dan W&B

Setiap seed selalu dimulai dari `yolo26n.pt` kecuali resume eksplisit. Setelah training selesai, `best.pt` dan `last.pt` diunggah ke W&B sebelum evaluasi dimulai. Ini menjadi *safe point* jika cell evaluasi berikutnya gagal.


In [ ]:
from ultralytics import YOLO, settings

settings.update({"wandb": False})


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def release_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_upload_checkpoint(run, checkpoint, seed, epoch, aliases):
    """Upload checkpoint ke W&B tanpa membuat network hiccup merusak training."""
    checkpoint = Path(checkpoint)

    if not WANDB_ENABLED or not checkpoint.exists():
        return False

    try:
        artifact = wandb.Artifact(
            f"yolo26n-seed-{seed}-checkpoint",
            type="model",
            metadata={
                "seed": seed,
                "epoch": epoch,
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "mode": CFG["mode"],
                "checkpoint_name": checkpoint.name,
            },
        )
        artifact.add_file(str(checkpoint), name=checkpoint.name)
        logged = run.log_artifact(artifact, aliases=aliases)
        logged.wait()
        return True
    except Exception as exc:
        print(
            f"[WARN] Upload {checkpoint.name} ke W&B gagal: "
            f"{type(exc).__name__}: {exc}"
        )
        return False


def train_one_seed(seed, resume_checkpoint=None):
    if not GPU_AVAILABLE:
        raise RuntimeError("GPU CUDA tidak tersedia.")

    set_seed(seed)
    release_gpu_memory()

    kind = "smoke" if IS_SMOKE else "train"
    run_name = f"yolo26n-baseline-{kind}-seed-{seed}"
    run_dir = RUNS_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    prior_run_id = None
    if resume_checkpoint:
        prior_metadata = Path(resume_checkpoint).parents[1] / "run_metadata.json"
        if prior_metadata.exists():
            prior_run_id = json.loads(
                prior_metadata.read_text(encoding="utf-8")
            ).get("wandb_run_id")

    run = wandb.init(
        project=CFG["wandb_project"],
        entity=CFG["wandb_entity"],
        name=run_name,
        group="yolo26n-fpnpan-baseline",
        job_type=kind,
        config={
            **CFG,
            "seed": seed,
            "dataset_fingerprint": DATASET_FINGERPRINT,
            "active_data_yaml": str(ACTIVE_DATA_YAML),
            **ENV_INFO,
        },
        id=prior_run_id,
        resume="allow" if prior_run_id else None,
        mode="online" if WANDB_ENABLED else "disabled",
        reinit="finish_previous",
    )

    model = YOLO(str(resume_checkpoint) if resume_checkpoint else CFG["model"])

    def on_fit_epoch_end(trainer):
        if not (WANDB_ENABLED and wandb.run):
            return

        logged_metrics = {}
        for key, value in trainer.metrics.items():
            if not np.isscalar(value):
                continue
            metric_key = key if key.startswith("val/") else f"val/{key}"
            logged_metrics[metric_key] = float(value)

        logged_metrics["epoch"] = trainer.epoch + 1
        wandb.log(logged_metrics)

    def on_model_save(trainer):
        epoch = trainer.epoch + 1
        if WANDB_ENABLED and epoch % CFG["artifact_period"] == 0:
            safe_upload_checkpoint(
                run,
                trainer.last,
                seed,
                epoch,
                aliases=["latest", f"epoch-{epoch}"],
            )

    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    model.add_callback("on_model_save", on_model_save)

    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    status = "failed"
    caught_error = None

    try:
        if resume_checkpoint:
            model.train(resume=True)
        else:
            model.train(
                data=str(ACTIVE_DATA_YAML),
                epochs=CFG["smoke_epochs"] if IS_SMOKE else CFG["epochs"],
                fraction=CFG["smoke_train_fraction"] if IS_SMOKE else 1.0,
                imgsz=CFG["imgsz"],
                batch=CFG["train_batch"],
                device=DEVICE,
                workers=CFG["workers"],
                optimizer=CFG["optimizer"],
                lr0=CFG["lr0"],
                weight_decay=CFG["weight_decay"],
                warmup_epochs=CFG["warmup_epochs"],
                patience=CFG["patience"],
                amp=CFG["amp"],
                seed=seed,
                deterministic=True,
                freeze=None,
                flipud=CFG["flipud"],
                mixup=CFG["mixup"],
                cutmix=CFG["cutmix"],
                close_mosaic=CFG["close_mosaic"],
                save=True,
                save_period=CFG["save_period"],
                plots=True,
                val=True,
                project=str(RUNS_ROOT),
                name=run_name,
                exist_ok=True,
                verbose=True,
            )
        status = "completed"

    except torch.cuda.OutOfMemoryError as exc:
        caught_error = exc
        raise RuntimeError(
            "GPU OOM saat training. Turunkan CFG['train_batch'] secara manual."
        ) from exc

    except Exception as exc:
        caught_error = exc
        raise

    finally:
        elapsed = time.perf_counter() - started
        peak_vram = (
            torch.cuda.max_memory_allocated() / 1024**2
            if torch.cuda.is_available()
            else None
        )

        metadata = {
            "seed": seed,
            "kind": kind,
            "status": status,
            "dataset_fingerprint": DATASET_FINGERPRINT,
            "duration_seconds": elapsed,
            "training_peak_vram_mb": peak_vram,
            "wandb_run_id": getattr(run, "id", None),
            "config": CFG,
            "error": None
            if caught_error is None
            else f"{type(caught_error).__name__}: {caught_error}",
        }
        (run_dir / "run_metadata.json").write_text(
            json.dumps(metadata, indent=2),
            encoding="utf-8",
        )

        best_path = run_dir / "weights" / "best.pt"
        last_path = run_dir / "weights" / "last.pt"

        best_uploaded = safe_upload_checkpoint(
            run,
            best_path,
            seed,
            -1,
            aliases=["best", status],
        )
        last_uploaded = safe_upload_checkpoint(
            run,
            last_path,
            seed,
            -1,
            aliases=["latest", status],
        )

        print(
            "Checkpoint safe point:",
            {
                "best_exists": best_path.exists(),
                "last_exists": last_path.exists(),
                "best_wandb": best_uploaded,
                "last_wandb": last_uploaded,
            },
        )

        run.finish()
        release_gpu_memory()

    return run_dir


## K. Inspeksi arsitektur dan profiling YOLO26n

Bagian ini dijalankan **sebelum training** untuk mendokumentasikan model kontrol yang benar-benar dipakai. `model.info(detailed=True)` menampilkan ringkasan layer, parameter, dan GFLOPs; struktur `model.model` ditampilkan agar backbone–neck–head dapat diperiksa. Notebook juga melakukan profiling per-layer dengan batch 1. Profiling ini hanya untuk inspeksi arsitektur dan **bukan** angka latency final penelitian.

Implementasi dibuat kompatibel: bila objek YOLO memiliki method `profile()`, method tersebut dicoba; bila tidak, notebook memakai profiling per-layer pada core detection model melalui `predict(..., profile=True)`.


In [ ]:
# ============================================================
# K. MODEL ARCHITECTURE & DIAGNOSTIC PROFILING
# ============================================================

def architecture_table(yolo_model):
    """Bangun tabel ringkas layer dari core Ultralytics DetectionModel."""
    rows = []

    for layer in yolo_model.model.model:
        parameter_count = getattr(layer, "np", None)
        if parameter_count is None:
            parameter_count = sum(p.numel() for p in layer.parameters())

        rows.append(
            {
                "index": int(getattr(layer, "i", len(rows))),
                "from": str(getattr(layer, "f", -1)),
                "module": str(
                    getattr(layer, "type", layer.__class__.__name__)
                ),
                "parameters": int(parameter_count),
            }
        )

    return pd.DataFrame(rows)


def profile_yolo26_architecture(model_path, imgsz):
    """
    Dokumentasikan arsitektur tanpa memengaruhi model yang akan ditraining.

    Returns
    -------
    dict
        Ringkasan model dari model.info().
    """
    release_gpu_memory()

    profile_model = YOLO(model_path)
    print("=" * 72)
    print("YOLO26n MODEL SUMMARY")
    print("=" * 72)

    info_result = profile_model.info(
        detailed=True,
        verbose=True,
        imgsz=imgsz,
    )

    print()
    print("=" * 72)
    print("PYTORCH MODEL STRUCTURE")
    print("=" * 72)
    print(profile_model.model)

    arch_df = architecture_table(profile_model)
    arch_path = REPORTS_ROOT / "model_architecture.csv"
    arch_df.to_csv(arch_path, index=False)

    summary = {
        "model": str(model_path),
        "imgsz": int(imgsz),
        "profile_backend": None,
    }

    if isinstance(info_result, (tuple, list)) and len(info_result) >= 4:
        summary.update(
            {
                "layers": int(info_result[0]),
                "parameters": int(info_result[1]),
                "gradients": int(info_result[2]),
                "gflops": float(info_result[3]),
            }
        )

    print()
    print("=" * 72)
    print("PER-LAYER DIAGNOSTIC PROFILE")
    print("=" * 72)

    try:
        wrapper_profile = getattr(profile_model, "profile", None)

        if callable(wrapper_profile):
            # Beberapa versi/build dapat menyediakan helper profile langsung.
            try:
                wrapper_profile()
                summary["profile_backend"] = "YOLO.profile()"
            except TypeError:
                wrapper_profile(imgsz=imgsz)
                summary["profile_backend"] = "YOLO.profile(imgsz=...)"
        else:
            # Ultralytics core DetectionModel mendukung predict(profile=True),
            # yang mencetak time/GFLOPs/params per layer.
            profile_device = torch.device(
                "cuda:0" if GPU_AVAILABLE else "cpu"
            )
            core = profile_model.model.to(profile_device).eval()
            dummy = torch.zeros(
                (1, 3, imgsz, imgsz),
                device=profile_device,
            )

            with torch.inference_mode():
                core.predict(dummy, profile=True)

            summary["profile_backend"] = (
                "DetectionModel.predict(profile=True)"
            )

    except Exception as exc:
        # Profiling diagnostik tidak boleh menggagalkan eksperimen.
        summary["profile_backend"] = "failed"
        summary["profile_error"] = f"{type(exc).__name__}: {exc}"
        print(
            "[WARN] Profiling per-layer gagal, tetapi model.info() "
            "dan tabel arsitektur tetap tersedia."
        )
        print(summary["profile_error"])

    summary_path = REPORTS_ROOT / "model_profile_summary.json"
    summary_path.write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    display(arch_df)
    print("Architecture CSV :", arch_path)
    print("Profile summary  :", summary_path)

    del profile_model
    release_gpu_memory()

    return summary


MODEL_PROFILE = profile_yolo26_architecture(
    CFG["model"],
    CFG["imgsz"],
)


## L. Jalankan training mode aktif

Pada notebook yang dibagikan ini, mode default adalah `smoke`, sehingga hanya satu epoch yang dijalankan. Setelah pipeline end-to-end lolos, ubah `CFG["mode"]` menjadi `research` dan Run All pada version baru.


In [ ]:
TRAIN_DIRS = {}

for seed in SEEDS_TO_RUN:
    TRAIN_DIRS[seed] = train_one_seed(seed)

    best_path = TRAIN_DIRS[seed] / "weights" / "best.pt"
    last_path = TRAIN_DIRS[seed] / "weights" / "last.pt"

    if not best_path.exists() or not last_path.exists():
        raise RuntimeError(
            f"Training seed {seed} selesai tanpa best.pt/last.pt yang lengkap."
        )

    print(f"Seed {seed} training selesai: {TRAIN_DIRS[seed]}")

print("SAFE POINT: seluruh checkpoint training tersedia sebelum evaluasi.")


### Resume training (opsional)

Jangan aktifkan pada smoke test normal. Resume hanya dipakai saat training research terputus dan checkpoint serta fingerprint dataset sudah diverifikasi.


In [ ]:
RUN_RESUME = False

if RUN_RESUME:
    if IS_SMOKE:
        raise RuntimeError("Resume tidak dipakai pada mode smoke.")

    resume_path = Path(CFG["resume_checkpoint"] or "")
    if not resume_path.exists():
        raise FileNotFoundError("Checkpoint resume tidak ditemukan.")

    metadata_path = resume_path.parents[1] / "run_metadata.json"
    if not metadata_path.exists():
        raise RuntimeError("run_metadata.json tidak ditemukan.")

    old = json.loads(metadata_path.read_text(encoding="utf-8"))
    if old.get("dataset_fingerprint") != DATASET_FINGERPRINT:
        raise RuntimeError("Fingerprint dataset berbeda; resume dibatalkan.")

    resume_seed = int(old["seed"])
    TRAIN_DIRS[resume_seed] = train_one_seed(
        resume_seed,
        resume_checkpoint=resume_path,
    )
else:
    print("Resume tidak dijalankan.")


## M. Evaluasi utama

Logika evaluasi tetap sama: threshold confidence dipilih **hanya dari validation**, lalu dikunci untuk test. Perubahan teknis utama adalah `eval_batch=1` dan pembersihan VRAM antartahap agar tidak mengulang OOM sebelumnya.


In [ ]:
def image_records(split):
    df = AUDIT_DF[
        (AUDIT_DF["split"] == split)
        & (AUDIT_DF["status"].isin(NON_BLOCKING_STATUS))
    ].copy()

    if IS_SMOKE and split in ACTIVE_EVAL_PATHS:
        allowed = ACTIVE_EVAL_PATHS[split]
        if allowed is not None:
            resolved = df["image"].map(canonical_path)
            df = df[resolved.isin(allowed)].copy()

    return list(df.itertuples(index=False))


def predict_records(model, split, conf=None):
    """Prediksi confidence rendah; batch evaluasi dipisahkan dari batch training."""
    conf = CFG["prediction_conf"] if conf is None else conf
    rows = image_records(split)
    paths = [record.image for record in rows]

    if not paths:
        raise RuntimeError(f"Tidak ada image aktif untuk split '{split}'.")

    output = {}

    for result in model.predict(
        paths,
        imgsz=CFG["imgsz"],
        conf=conf,
        iou=CFG["prediction_iou"],
        nms=False,
        device=DEVICE,
        batch=CFG["eval_batch"],
        verbose=False,
        stream=True,
    ):
        boxes = result.boxes
        key = canonical_path(result.path)

        if boxes is None:
            output[key] = []
            continue

        output[key] = [
            {
                "cls": int(cls),
                "conf": float(score),
                "xyxy": [float(value) for value in box],
            }
            for box, score, cls in zip(
                boxes.xyxy.cpu().numpy(),
                boxes.conf.cpu().numpy(),
                boxes.cls.cpu().numpy(),
            )
        ]

    return output


def iou_one_to_many(box, boxes):
    if len(boxes) == 0:
        return np.empty(0)

    boxes = np.asarray(boxes, dtype=float)
    box = np.asarray(box, dtype=float)

    intersection_size = np.maximum(
        0,
        np.minimum(box[2:], boxes[:, 2:])
        - np.maximum(box[:2], boxes[:, :2]),
    )
    intersection = intersection_size[:, 0] * intersection_size[:, 1]
    area_box = np.prod(box[2:] - box[:2])
    area_boxes = np.prod(boxes[:, 2:] - boxes[:, :2], axis=1)

    return intersection / np.maximum(
        area_box + area_boxes - intersection,
        1e-9,
    )


def match_class_boxes(gt_boxes, predictions, threshold, iou_threshold=0.5):
    """Greedy matching satu kelas. Output: TP, FP, FN."""
    matched = set()
    tp = 0
    fp = 0

    selected = sorted(
        [prediction for prediction in predictions if prediction["conf"] >= threshold],
        key=lambda item: item["conf"],
        reverse=True,
    )

    for prediction in selected:
        available = [
            index for index in range(len(gt_boxes)) if index not in matched
        ]
        candidate_boxes = [gt_boxes[index] for index in available]
        ious = iou_one_to_many(prediction["xyxy"], candidate_boxes)

        if len(ious) and ious.max() >= iou_threshold:
            matched.add(available[int(ious.argmax())])
            tp += 1
        else:
            fp += 1

    return tp, fp, len(gt_boxes) - len(matched)


def detection_counts(split, predictions, threshold, iou_threshold=0.5):
    counts = np.zeros((len(names), 3), dtype=int)

    for record in image_records(split):
        key = canonical_path(record.image)
        ground_truth = read_yolo_boxes(
            record.label,
            record.width,
            record.height,
        )

        for class_id in range(len(names)):
            gt_boxes = [
                item[1:] for item in ground_truth if item[0] == class_id
            ]
            class_predictions = [
                item
                for item in predictions.get(key, [])
                if item["cls"] == class_id
            ]
            counts[class_id] += match_class_boxes(
                gt_boxes,
                class_predictions,
                threshold,
                iou_threshold,
            )

    return counts


def counts_to_frame(counts):
    rows = []

    for class_id, (tp, fp, fn) in enumerate(counts):
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        rows.append(
            {
                "class_id": class_id,
                "class_name": names[class_id],
                "TP": int(tp),
                "FP": int(fp),
                "FN": int(fn),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "ground_truth": int(tp + fn),
            }
        )

    return pd.DataFrame(rows)


def summarize_counts(counts):
    frame = counts_to_frame(counts)
    tp, fp, fn = counts.sum(axis=0)

    micro_precision = tp / (tp + fp) if tp + fp else 0.0
    micro_recall = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall
        / (micro_precision + micro_recall)
        if micro_precision + micro_recall
        else 0.0
    )

    valid = frame[frame["ground_truth"] > 0]

    return {
        "precision": float(micro_precision),
        "recall": float(micro_recall),
        "f1": float(micro_f1),
        "macro_precision": float(valid["precision"].mean()) if len(valid) else np.nan,
        "macro_recall": float(valid["recall"].mean()) if len(valid) else np.nan,
        "macro_f1": float(valid["f1"].mean()) if len(valid) else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
    }


def select_threshold(validation_predictions):
    candidates = np.linspace(0.01, 0.99, 99)
    scores = []

    for threshold in candidates:
        frame = counts_to_frame(
            detection_counts("val", validation_predictions, threshold)
        )
        valid = frame[frame["ground_truth"] > 0]
        scores.append(float(valid["f1"].mean()) if len(valid) else 0.0)

    best_index = int(np.argmax(scores))
    return float(candidates[best_index]), pd.DataFrame(
        {"confidence": candidates, "macro_f1": scores}
    )


def save_dataframe(frame, csv_path):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(csv_path, index=False)
    frame.to_json(
        csv_path.with_suffix(".json"),
        orient="records",
        indent=2,
        force_ascii=False,
    )


def save_figure(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()


### Self-test evaluator TP/FP/FN


In [ ]:
perfect = match_class_boxes([[0, 0, 10, 10]], [{"xyxy": [0, 0, 10, 10], "conf": 0.9}], 0.5)
empty_prediction = match_class_boxes([[0, 0, 10, 10]], [], 0.5)
false_positive = match_class_boxes([], [{"xyxy": [0, 0, 10, 10], "conf": 0.9}], 0.5)
wrong_location = match_class_boxes([[0, 0, 10, 10]], [{"xyxy": [20, 20, 30, 30], "conf": 0.9}], 0.5)
assert perfect == (1, 0, 0)
assert empty_prediction == (0, 0, 1)
assert false_positive == (0, 1, 0)
assert wrong_location == (0, 1, 1)

# Prediksi kelas 1 terhadap GT kelas 0 berarti FN pada kelas 0 dan FP pada kelas 1.
wrong_class_counts = np.array([
    match_class_boxes([[0, 0, 10, 10]], [], 0.5),
    match_class_boxes([], [{"xyxy": [0, 0, 10, 10], "conf": 0.9}], 0.5),
])
assert tuple(wrong_class_counts[0]) == (0, 0, 1)
assert tuple(wrong_class_counts[1]) == (0, 1, 0)
print("Self-test matching TP/FP/FN: LULUS")


### Jalankan evaluasi validation → test


In [ ]:
EVAL_RESULTS = {}

for seed in SEEDS_TO_RUN:
    run_dir = TRAIN_DIRS.get(seed)
    if run_dir is None:
        raise RuntimeError(f"Run directory untuk seed {seed} tidak tersedia.")

    best_path = run_dir / "weights" / "best.pt"
    if not best_path.exists():
        raise FileNotFoundError(f"best.pt tidak ditemukan: {best_path}")

    release_gpu_memory()
    model = YOLO(str(best_path))

    eval_dir = run_dir / "evaluation"
    figure_dir = eval_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # 1. Select confidence hanya dari validation
    # --------------------------------------------------------
    val_predictions = predict_records(
        model,
        "val",
        conf=CFG["prediction_conf"],
    )
    best_confidence, threshold_curve = select_threshold(val_predictions)
    save_dataframe(
        threshold_curve,
        eval_dir / "validation_threshold_curve.csv",
    )

    del val_predictions
    release_gpu_memory()

    # --------------------------------------------------------
    # 2. Test pada threshold yang sudah dikunci
    # --------------------------------------------------------
    test_predictions = predict_records(
        model,
        "test",
        conf=CFG["prediction_conf"],
    )
    test_counts = detection_counts(
        "test",
        test_predictions,
        best_confidence,
    )
    test_per_class = counts_to_frame(test_counts)
    count_summary = summarize_counts(test_counts)
    save_dataframe(
        test_per_class,
        eval_dir / "test_per_class_operating_point.csv",
    )

    # --------------------------------------------------------
    # 3. Ultralytics AP/mAP pada test
    # --------------------------------------------------------
    release_gpu_memory()
    metrics = model.val(
        data=str(ACTIVE_DATA_YAML),
        split="test",
        imgsz=CFG["imgsz"],
        batch=CFG["eval_batch"],
        conf=CFG["prediction_conf"],
        iou=CFG["prediction_iou"],
        nms=False,
        device=DEVICE,
        plots=True,
        project=str(eval_dir),
        name="ultralytics",
        exist_ok=True,
        verbose=False,
    )

    ultralytics_dir = Path(metrics.save_dir)
    ap50_all = np.full(len(names), np.nan, dtype=float)
    ap_class_index = np.asarray(metrics.box.ap_class_index, dtype=int)
    ap50_present = np.asarray(metrics.box.ap50, dtype=float)
    if len(ap_class_index) == len(ap50_present):
        ap50_all[ap_class_index] = ap50_present

    ap50_95_all = np.asarray(metrics.box.maps, dtype=float)
    if len(ap50_95_all) != len(names):
        fallback = np.full(len(names), np.nan, dtype=float)
        ap_present = np.asarray(metrics.box.ap, dtype=float)
        if len(ap_class_index) == len(ap_present):
            fallback[ap_class_index] = ap_present
        ap50_95_all = fallback

    per_class_ap = pd.DataFrame(
        {
            "class_id": range(len(names)),
            "class_name": names,
            "AP50": ap50_all,
            "AP50_95": ap50_95_all,
        }
    ).merge(
        test_per_class[["class_id", "ground_truth"]],
        on="class_id",
        how="left",
    )
    save_dataframe(per_class_ap, eval_dir / "test_per_class_ap.csv")

    overall = {
        "architecture": "FPN-PAN",
        "model": "YOLO26n FPN-PAN",
        "mode": CFG["mode"],
        "seed": seed,
        "confidence_from_validation": best_confidence,
        **count_summary,
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "ultralytics_precision_diagnostic": float(metrics.box.mp),
        "ultralytics_recall_diagnostic": float(metrics.box.mr),
        "dataset_fingerprint": DATASET_FINGERPRINT,
    }
    (eval_dir / "ultralytics_overall.json").write_text(
        json.dumps(overall, indent=2),
        encoding="utf-8",
    )

    table_35 = pd.DataFrame(
        [
            {
                "Model": "YOLO26n FPN-PAN",
                "Seed": seed,
                "mAP50-95": overall["mAP50_95"],
                "mAP50": overall["mAP50"],
                "Precision": overall["precision"],
                "Recall": overall["recall"],
                "F1-score": overall["f1"],
            }
        ]
    )
    save_dataframe(table_35, eval_dir / "tabel_3_5_kinerja_keseluruhan.csv")

    table_36 = (
        per_class_ap.set_index("class_name")["AP50_95"]
        .reindex(names)
        .to_frame()
        .T
    )
    table_36.insert(0, "Model", "YOLO26n FPN-PAN")
    table_36.insert(1, "Seed", seed)
    save_dataframe(table_36, eval_dir / "tabel_3_6_ap_per_kelas.csv")

    table_36_ap50 = (
        per_class_ap.set_index("class_name")["AP50"]
        .reindex(names)
        .to_frame()
        .T
    )
    table_36_ap50.insert(0, "Model", "YOLO26n FPN-PAN")
    table_36_ap50.insert(1, "Seed", seed)
    save_dataframe(
        table_36_ap50,
        eval_dir / "tabel_pendukung_ap50_per_kelas.csv",
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(threshold_curve["confidence"], threshold_curve["macro_f1"])
    ax.axvline(best_confidence, linestyle="--", label=f"Terpilih = {best_confidence:.2f}")
    ax.set(
        title="Pemilihan Confidence dari Validation Set",
        xlabel="Confidence",
        ylabel="Macro F1",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    save_figure(fig, figure_dir / "validation_macro_f1_threshold.png")

    positions = np.arange(len(names))
    width = 0.36
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(positions - width / 2, per_class_ap["AP50"], width, label="AP50")
    ax.bar(
        positions + width / 2,
        per_class_ap["AP50_95"],
        width,
        label="AP50-95",
    )
    ax.set_xticks(positions, names, rotation=25, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Average Precision")
    ax.set_title("AP per Kelas pada Test Set")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="upper left")

    count_axis = ax.twinx()
    count_axis.plot(
        positions,
        per_class_ap["ground_truth"],
        marker="o",
        linestyle="--",
        label="Ground truth",
    )
    count_axis.set_ylabel("Jumlah ground truth")
    save_figure(fig, figure_dir / "ap_per_kelas_dan_jumlah_gt.png")

    for filename, title in [
        ("confusion_matrix.png", "Confusion Matrix Absolut"),
        ("confusion_matrix_normalized.png", "Confusion Matrix Normalized"),
    ]:
        source = ultralytics_dir / filename
        if source.exists():
            target = figure_dir / filename
            shutil.copy2(source, target)
            print(title)
            display(Image.open(target))

    EVAL_RESULTS[seed] = {
        "model": model,
        "run_dir": run_dir,
        "eval_dir": eval_dir,
        "figure_dir": figure_dir,
        "predictions": test_predictions,
        "overall": overall,
        "per_class_ap": per_class_ap,
    }

    display(table_35.style.format(precision=4))
    display(table_36.style.format(precision=4))
    display(table_36_ap50.style.format(precision=4))
    display(
        test_per_class.style.format(
            {
                "precision": "{:.4f}",
                "recall": "{:.4f}",
                "f1": "{:.4f}",
            }
        )
    )

    print(f"Seed {seed}: evaluasi utama selesai.")


### Grafik proses training


In [ ]:
for seed, item in EVAL_RESULTS.items():
    results_csv = item["run_dir"] / "results.csv"
    if not results_csv.exists():
        print(f"Seed {seed}: results.csv tidak tersedia; grafik training dilewati.")
        continue
    history = pd.read_csv(results_csv)
    history.columns = [column.strip() for column in history.columns]
    panels = [
        ("train/box_loss", "Training box loss"),
        ("val/box_loss", "Validation box loss"),
        ("metrics/mAP50(B)", "mAP50 validation"),
        ("metrics/mAP50-95(B)", "mAP50-95 validation"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for ax, (column, title) in zip(axes.ravel(), panels):
        if column in history:
            ax.plot(np.arange(1, len(history) + 1), history[column], linewidth=2)
        else:
            ax.text(0.5, 0.5, f"Kolom {column} tidak tersedia", ha="center", va="center")
        ax.set(title=title, xlabel="Epoch"); ax.grid(alpha=0.25)
    fig.suptitle(f"Riwayat Training YOLO26n FPN-PAN - Seed {seed}")
    fig.tight_layout()
    save_figure(fig, item["figure_dir"] / "training_curves.png")


## N. Evaluasi COCO berdasarkan skala

Logika bagian ini dipertahankan dari notebook sebelumnya: COCO evaluator digunakan untuk AP small/medium/large dan hasil disimpan terpisah dari metrik Ultralytics.


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def build_coco(split, predictions):
    dataset = {
        "info": {}, "licenses": [],
        "images": [], "annotations": [],
        "categories": [{"id": index + 1, "name": name} for index, name in enumerate(names)],
    }
    results = []
    annotation_id = 1
    for image_id, record in enumerate(image_records(split), 1):
        dataset["images"].append({
            "id": image_id, "file_name": Path(record.image).name,
            "width": record.width, "height": record.height,
        })
        for cls, x1, y1, x2, y2 in read_yolo_boxes(
            record.label, record.width, record.height
        ):
            width, height = x2 - x1, y2 - y1
            dataset["annotations"].append({
                "id": annotation_id, "image_id": image_id, "category_id": cls + 1,
                "bbox": [x1, y1, width, height], "area": width * height, "iscrowd": 0,
            })
            annotation_id += 1
        key = str(Path(record.image).resolve())
        for prediction in predictions.get(key, []):
            x1, y1, x2, y2 = prediction["xyxy"]
            results.append({
                "image_id": image_id, "category_id": prediction["cls"] + 1,
                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": prediction["conf"],
            })
    return dataset, results


def ground_truth_scale_counts(dataset):
    counts = {"small": 0, "medium": 0, "large": 0}
    for annotation in dataset["annotations"]:
        area = annotation["area"]
        key = "small" if area < 32**2 else "medium" if area < 96**2 else "large"
        counts[key] += 1
    return counts


def coco_scale_eval(dataset, results, quiet=False):
    counts = ground_truth_scale_counts(dataset)
    if not results:
        return {
            "AP_small": 0.0 if counts["small"] else np.nan,
            "AP_medium": 0.0 if counts["medium"] else np.nan,
            "AP_large": 0.0 if counts["large"] else np.nan,
        }, None
    coco_gt = COCO()
    coco_gt.dataset = dataset
    coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(results)
    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    values = evaluator.stats[3:6]
    output = {}
    for key, value, count in zip(
        ["AP_small", "AP_medium", "AP_large"], values, counts.values()
    ):
        output[key] = float(value) if count and value >= 0 else np.nan
    return output, evaluator


# Self-test batas skala dan evaluator sempurna.
assert ground_truth_scale_counts({"annotations": [
    {"area": 32**2 - 1}, {"area": 32**2}, {"area": 96**2 - 1}, {"area": 96**2},
]}) == {"small": 1, "medium": 2, "large": 1}

synthetic_dataset = {
    "info": {}, "licenses": [],
    "images": [
        {"id": 1, "file_name": "small.jpg", "width": 200, "height": 200},
        {"id": 2, "file_name": "medium.jpg", "width": 200, "height": 200},
        {"id": 3, "file_name": "large.jpg", "width": 200, "height": 200},
    ],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1, "bbox": [0, 0, 16, 16], "area": 256, "iscrowd": 0},
        {"id": 2, "image_id": 2, "category_id": 1, "bbox": [0, 0, 64, 64], "area": 4096, "iscrowd": 0},
        {"id": 3, "image_id": 3, "category_id": 1, "bbox": [0, 0, 128, 128], "area": 16384, "iscrowd": 0},
    ],
    "categories": [{"id": 1, "name": "objek"}],
}
synthetic_predictions = [
    {"image_id": annotation["image_id"], "category_id": 1,
     "bbox": annotation["bbox"], "score": 0.99}
    for annotation in synthetic_dataset["annotations"]
]
perfect_scale, _ = coco_scale_eval(synthetic_dataset, synthetic_predictions)
assert all(value > 0.99 for value in perfect_scale.values())
empty_scale, _ = coco_scale_eval(synthetic_dataset, [])
assert empty_scale == {"AP_small": 0.0, "AP_medium": 0.0, "AP_large": 0.0}
print("Self-test COCO dan batas skala: LULUS")

for seed, item in EVAL_RESULTS.items():
    coco_gt, coco_predictions = build_coco("test", item["predictions"])
    scale_metrics, evaluator = coco_scale_eval(coco_gt, coco_predictions)
    item["overall"].update(scale_metrics)
    (item["eval_dir"] / "coco_ground_truth.json").write_text(json.dumps(coco_gt), encoding="utf-8")
    (item["eval_dir"] / "coco_predictions.json").write_text(json.dumps(coco_predictions), encoding="utf-8")
    (item["eval_dir"] / "coco_scale_metrics.json").write_text(
        json.dumps(scale_metrics, indent=2, allow_nan=True), encoding="utf-8"
    )
    table_37 = pd.DataFrame([{
        "Model": "YOLO26n FPN-PAN", "Seed": seed,
        "AP-S": scale_metrics["AP_small"],
        "AP-M": scale_metrics["AP_medium"],
        "AP-L": scale_metrics["AP_large"],
    }])
    save_dataframe(table_37, item["eval_dir"] / "tabel_3_7_ap_berdasarkan_skala.csv")

    fig, ax = plt.subplots(figsize=(8, 5))
    values = [scale_metrics["AP_small"], scale_metrics["AP_medium"], scale_metrics["AP_large"]]
    bars = ax.bar(["Small", "Medium", "Large"], values)
    ax.set(title="AP Berdasarkan Skala Objek", ylabel="AP50-95", ylim=(0, 1))
    ax.grid(axis="y", alpha=0.25)
    for bar, value in zip(bars, values):
        if not pd.isna(value): ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:.3f}", ha="center", va="bottom")
    save_figure(fig, item["figure_dir"] / "ap_berdasarkan_skala.png")
    display(table_37.style.format(precision=4))


## O. Efisiensi komputasi

Research mode tetap memakai 50 warmup + 200 pengukuran. Smoke mode memakai jumlah lebih kecil untuk memastikan jalur benchmark bekerja tanpa membuang waktu.


In [ ]:
def benchmark_model(model, sample_path):
    if not GPU_AVAILABLE:
        raise RuntimeError("Benchmark GPU membutuhkan CUDA.")

    release_gpu_memory()

    core = model.model.to("cuda").eval()
    use_half = bool(CFG["amp"])
    core = core.half() if use_half else core.float()

    dtype = torch.float16 if use_half else torch.float32
    dummy = torch.zeros(
        1,
        3,
        CFG["imgsz"],
        CFG["imgsz"],
        device="cuda",
        dtype=dtype,
    )

    warmup = (
        CFG["smoke_benchmark_warmup"]
        if IS_SMOKE
        else CFG["benchmark_warmup"]
    )
    repeats = (
        CFG["smoke_benchmark_runs"]
        if IS_SMOKE
        else CFG["benchmark_runs"]
    )

    torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        for _ in range(warmup):
            core(dummy)

        torch.cuda.synchronize()
        forward_times = []

        for _ in range(repeats):
            start = time.perf_counter()
            core(dummy)
            torch.cuda.synchronize()
            forward_times.append((time.perf_counter() - start) * 1000)

    pipeline_times = []

    for _ in range(warmup):
        model.predict(
            sample_path,
            imgsz=CFG["imgsz"],
            device=0,
            half=use_half,
            nms=False,
            verbose=False,
        )

    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        model.predict(
            sample_path,
            imgsz=CFG["imgsz"],
            device=0,
            half=use_half,
            nms=False,
            verbose=False,
        )
        torch.cuda.synchronize()
        pipeline_times.append((time.perf_counter() - start) * 1000)

    params = sum(parameter.numel() for parameter in core.parameters())

    try:
        from ultralytics.utils.torch_utils import get_flops
        gflops = float(get_flops(core, imgsz=CFG["imgsz"]))
    except Exception:
        gflops = np.nan

    return {
        "parameters": params,
        "gflops": gflops,
        "checkpoint_mb": None,
        "forward_latency_ms_mean": float(np.mean(forward_times)),
        "forward_latency_ms_std": float(np.std(forward_times, ddof=1))
        if len(forward_times) > 1
        else 0.0,
        "forward_fps": 1000 / float(np.mean(forward_times)),
        "pipeline_latency_ms_mean": float(np.mean(pipeline_times)),
        "pipeline_latency_ms_std": float(np.std(pipeline_times, ddof=1))
        if len(pipeline_times) > 1
        else 0.0,
        "pipeline_fps": 1000 / float(np.mean(pipeline_times)),
        "peak_vram_mb": torch.cuda.max_memory_allocated() / 1024**2,
        "gpu": GPU_NAME,
        "imgsz": CFG["imgsz"],
        "batch": 1,
        "precision": "FP16" if use_half else "FP32",
        "backend": "PyTorch",
        "benchmark_warmup": warmup,
        "benchmark_runs": repeats,
    }


sample_path = image_records("test")[0].image

for seed, item in EVAL_RESULTS.items():
    efficiency = benchmark_model(item["model"], sample_path)
    efficiency["checkpoint_mb"] = (
        item["run_dir"] / "weights" / "best.pt"
    ).stat().st_size / 1024**2

    metadata = json.loads(
        (item["run_dir"] / "run_metadata.json").read_text(encoding="utf-8")
    )
    efficiency["training_duration_seconds"] = metadata["duration_seconds"]
    efficiency["training_peak_vram_mb"] = metadata.get("training_peak_vram_mb")

    item["overall"].update(efficiency)
    (item["eval_dir"] / "efficiency.json").write_text(
        json.dumps(efficiency, indent=2),
        encoding="utf-8",
    )

    table_38 = pd.DataFrame(
        [
            {
                "Model": "YOLO26n FPN-PAN",
                "Seed": seed,
                "Parameters (juta)": efficiency["parameters"] / 1e6,
                "GFLOPs": efficiency["gflops"],
                "Model size (MB)": efficiency["checkpoint_mb"],
                "Waktu training (jam)": efficiency["training_duration_seconds"] / 3600,
                "Latency (ms/citra)": efficiency["forward_latency_ms_mean"],
                "FPS": efficiency["forward_fps"],
                "Peak VRAM training (MB)": efficiency["training_peak_vram_mb"],
                "Peak VRAM inferensi (MB)": efficiency["peak_vram_mb"],
                "GPU": efficiency["gpu"],
                "Precision": efficiency["precision"],
                "Resolusi": efficiency["imgsz"],
                "Backend": efficiency["backend"],
                "Pipeline latency (ms/citra)": efficiency["pipeline_latency_ms_mean"],
                "Pipeline FPS": efficiency["pipeline_fps"],
            }
        ]
    )
    save_dataframe(
        table_38,
        item["eval_dir"] / "tabel_3_8_efisiensi_komputasi.csv",
    )
    display(table_38.style.format(precision=4))

release_gpu_memory()


## P. Contoh TP, FP, dan FN


In [ ]:
def save_error_examples(seed, item, limit=6):
    out_dir = item["eval_dir"] / "error_examples"; out_dir.mkdir(exist_ok=True)
    threshold = item["overall"]["confidence_from_validation"]; saved = 0
    for r in image_records("test"):
        im = Image.open(r.image).convert("RGB"); draw = ImageDraw.Draw(im)
        gt = read_yolo_boxes(r.label, r.width, r.height); matched = set(); has_error = False
        preds = sorted([p for p in item["predictions"].get(str(Path(r.image).resolve()), []) if p["conf"] >= threshold],
                       key=lambda x: x["conf"], reverse=True)
        for pred in preds:
            candidates = [i for i, g in enumerate(gt) if i not in matched and g[0] == pred["cls"]]
            ious = iou_one_to_many(pred["xyxy"], [gt[i][1:] for i in candidates])
            if len(ious) and ious.max() >= 0.5:
                matched.add(candidates[int(ious.argmax())]); color, tag = "lime", "TP"
            else: color, tag, has_error = "red", "FP", True
            draw.rectangle(pred["xyxy"], outline=color, width=3); draw.text(pred["xyxy"][:2], f"{tag} {names[pred['cls']]} {pred['conf']:.2f}", fill=color)
        for i, g in enumerate(gt):
            if i not in matched:
                has_error = True; draw.rectangle(g[1:], outline="orange", width=3); draw.text(g[1:3], f"FN {names[g[0]]}", fill="orange")
        if has_error:
            im.save(out_dir / f"{saved:02d}_{Path(r.image).name}"); saved += 1
            if saved >= limit: break
    print(f"Seed {seed}: {saved} contoh disimpan di {out_dir}")
    paths = sorted(out_dir.glob("*"))[:limit]
    if paths:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = np.atleast_1d(axes).ravel()
        for ax in axes: ax.axis("off")
        for ax, path in zip(axes, paths):
            ax.imshow(Image.open(path)); ax.set_title(path.name); ax.axis("off")
        fig.suptitle(f"Contoh TP (hijau), FP (merah), dan FN (oranye) - Seed {seed}")
        fig.tight_layout()
        save_figure(fig, item["figure_dir"] / "gallery_tp_fp_fn.png")

for seed, item in EVAL_RESULTS.items(): save_error_examples(seed, item)


## Q. Ringkasan run

Smoke mode hanya menyimpan ringkasan smoke. Ia **tidak** menulis hasil ke tabel komparasi antararsitektur dan tidak menjalankan Friedman. Research mode mempertahankan logika ringkasan/komparasi yang sudah ada.


In [ ]:
from scipy.stats import friedmanchisquare


def find_seed_evaluation(seed):
    # Prioritas run yang baru saja dibuat pada session ini.
    if seed in TRAIN_DIRS:
        current = TRAIN_DIRS[seed] / "evaluation"
        if (current / "ultralytics_overall.json").exists():
            return current

    # Fallback research run standar.
    local = RUNS_ROOT / f"yolo26n-baseline-train-seed-{seed}" / "evaluation"
    if (local / "ultralytics_overall.json").exists():
        return local

    # Fallback bundle hasil restore.
    if RESTORE_ROOT.exists():
        matches = list(
            RESTORE_ROOT.rglob(
                f"seed_{seed}/evaluation/ultralytics_overall.json"
            )
        )
        if matches:
            return matches[0].parent

    return None


completed = []
proposal_tables = {"3.5": [], "3.6": [], "3.7": [], "3.8": []}

for seed in SEEDS_TO_RUN if IS_SMOKE else CFG["seeds"]:
    eval_dir = find_seed_evaluation(seed)
    if eval_dir is None:
        continue

    overall = json.loads(
        (eval_dir / "ultralytics_overall.json").read_text(encoding="utf-8")
    )

    scale_path = eval_dir / "coco_scale_metrics.json"
    efficiency_path = eval_dir / "efficiency.json"

    if scale_path.exists():
        overall.update(json.loads(scale_path.read_text(encoding="utf-8")))
    if efficiency_path.exists():
        overall.update(json.loads(efficiency_path.read_text(encoding="utf-8")))

    completed.append(overall)

    for key, filename in {
        "3.5": "tabel_3_5_kinerja_keseluruhan.csv",
        "3.6": "tabel_3_6_ap_per_kelas.csv",
        "3.7": "tabel_3_7_ap_berdasarkan_skala.csv",
        "3.8": "tabel_3_8_efisiensi_komputasi.csv",
    }.items():
        path = eval_dir / filename
        if path.exists():
            proposal_tables[key].append(pd.read_csv(path))

SEED_SUMMARY = pd.DataFrame(completed)

if len(SEED_SUMMARY):
    summary_name = "smoke_results.csv" if IS_SMOKE else "seed_results.csv"
    save_dataframe(SEED_SUMMARY, REPORTS_ROOT / summary_name)
    display(SEED_SUMMARY)
else:
    raise RuntimeError("Belum ada seed dengan evaluasi lengkap.")

if IS_SMOKE:
    print(
        "SMOKE MODE: hasil tidak dimasukkan ke comparison_all_architectures "
        "dan Friedman tidak dijalankan."
    )
    AGG_SUMMARY = pd.DataFrame()

else:
    numeric = SEED_SUMMARY.select_dtypes(include=np.number).drop(
        columns=["seed"],
        errors="ignore",
    )
    AGG_SUMMARY = pd.DataFrame(
        {
            "mean": numeric.mean(),
            "sample_std": numeric.std(ddof=1),
            "completed_seeds": len(SEED_SUMMARY),
        }
    )
    AGG_SUMMARY.to_csv(REPORTS_ROOT / "seed_aggregate.csv")
    display(AGG_SUMMARY)

    if len(SEED_SUMMARY) >= 2:
        fig, ax = plt.subplots(figsize=(7, 5))
        values = SEED_SUMMARY["mAP50_95"]
        ax.scatter(SEED_SUMMARY["seed"].astype(str), values, s=90)
        ax.axhline(values.mean(), linestyle="--", label=f"Mean = {values.mean():.4f}")
        ax.set(
            title="Variasi mAP50-95 Antar-Seed",
            xlabel="Seed",
            ylabel="mAP50-95",
            ylim=(0, 1),
        )
        ax.grid(axis="y", alpha=0.25)
        ax.legend()
        save_figure(fig, REPORTS_ROOT / "variasi_map_antar_seed.png")

    for table_number, frames in proposal_tables.items():
        if frames:
            combined = pd.concat(frames, ignore_index=True)
            save_dataframe(
                combined,
                REPORTS_ROOT
                / f"tabel_{table_number.replace('.', '_')}_semua_seed.csv",
            )
            display(combined)

    comparison_columns = [
        "Arsitektur",
        "Seed",
        "mAP50-95",
        "Latency (ms/citra)",
        "Parameters (juta)",
        "GPU",
        "Precision komputasi",
        "Resolusi",
        "Dataset fingerprint",
    ]

    baseline_records = []
    for row in completed:
        baseline_records.append(
            {
                "Arsitektur": row.get("architecture", "FPN-PAN"),
                "Seed": row["seed"],
                "mAP50-95": row["mAP50_95"],
                "Latency (ms/citra)": row.get("forward_latency_ms_mean"),
                "Parameters (juta)": row.get("parameters", np.nan) / 1e6,
                "GPU": row.get("gpu"),
                "Precision komputasi": row.get("precision"),
                "Resolusi": row.get("imgsz", CFG["imgsz"]),
                "Dataset fingerprint": row.get(
                    "dataset_fingerprint",
                    DATASET_FINGERPRINT,
                ),
            }
        )

    comparison_path = REPORTS_ROOT / "comparison_all_architectures.csv"
    existing = (
        pd.read_csv(comparison_path)
        if comparison_path.exists()
        else pd.DataFrame(columns=comparison_columns)
    )
    comparison = pd.concat(
        [existing, pd.DataFrame(baseline_records)],
        ignore_index=True,
    ).drop_duplicates(["Arsitektur", "Seed"], keep="last")
    save_dataframe(comparison, comparison_path)

    consistency_columns = [
        "GPU",
        "Precision komputasi",
        "Resolusi",
        "Dataset fingerprint",
    ]
    consistency = {
        column: comparison[column].dropna().nunique()
        for column in consistency_columns
    }
    consistency_ok = all(value <= 1 for value in consistency.values())
    (REPORTS_ROOT / "comparison_consistency.json").write_text(
        json.dumps(
            {"consistent": consistency_ok, "unique_counts": consistency},
            indent=2,
        ),
        encoding="utf-8",
    )

    architectures = ["FPN-PAN", "BiFPN", "ASFF", "AFPN"]
    seeds = CFG["seeds"]
    pivot = comparison.pivot_table(
        index="Seed",
        columns="Arsitektur",
        values="mAP50-95",
        aggfunc="first",
    )

    complete = (
        all(architecture in pivot.columns for architecture in architectures)
        and all(seed in pivot.index for seed in seeds)
    )
    if complete:
        complete = not pivot.loc[seeds, architectures].isna().any().any()

    if complete:
        statistic, p_value = friedmanchisquare(
            *(pivot.loc[seeds, architecture] for architecture in architectures)
        )
        friedman_result = pd.DataFrame(
            [
                {
                    "Statistik Friedman": statistic,
                    "p-value": p_value,
                    "alpha": 0.05,
                    "Keputusan": "Tolak H0"
                    if p_value < 0.05
                    else "Belum menolak H0",
                }
            ]
        )
        save_dataframe(
            pivot.reset_index(),
            REPORTS_ROOT / "tabel_3_9_data_friedman.csv",
        )
        save_dataframe(
            friedman_result,
            REPORTS_ROOT / "hasil_friedman.csv",
        )
        display(pivot)
        display(friedman_result)
    else:
        status = {
            "status": "not_run",
            "reason": "Data 3 seed x 4 arsitektur belum lengkap",
            "missing_architectures": sorted(
                set(architectures) - set(pivot.columns)
            ),
        }
        (REPORTS_ROOT / "friedman_status.json").write_text(
            json.dumps(status, indent=2),
            encoding="utf-8",
        )
        print("Friedman Test belum dijalankan:", status["reason"])


## R. Upload evaluasi ke W&B

Gangguan upload W&B tidak membatalkan hasil evaluasi lokal. Status upload dicatat di report.


In [ ]:
WANDB_EVAL_UPLOAD_STATUS = []

if WANDB_ENABLED:
    for seed, item in EVAL_RESULTS.items():
        run = None
        try:
            eval_name = (
                f"yolo26n-fpnpan-evaluation-{CFG['mode']}-seed-{seed}"
            )
            run = wandb.init(
                project=CFG["wandb_project"],
                entity=CFG["wandb_entity"],
                name=eval_name,
                group="yolo26n-fpnpan-baseline",
                job_type=f"{CFG['mode']}-evaluation",
                config={
                    "architecture": "FPN-PAN",
                    "mode": CFG["mode"],
                    "seed": seed,
                    "dataset_fingerprint": DATASET_FINGERPRINT,
                    "imgsz": CFG["imgsz"],
                    "eval_batch": CFG["eval_batch"],
                    "nms": False,
                },
                reinit="finish_previous",
            )

            scalar_metrics = {
                key: value
                for key, value in item["overall"].items()
                if isinstance(value, (int, float)) and not pd.isna(value)
            }
            run.log(
                {f"test/{key}": value for key, value in scalar_metrics.items()}
            )

            for csv_path in item["eval_dir"].glob("*.csv"):
                run.log(
                    {
                        f"tables/{csv_path.stem}": wandb.Table(
                            dataframe=pd.read_csv(csv_path)
                        )
                    }
                )

            for image_path in item["figure_dir"].glob("*.png"):
                run.log(
                    {
                        f"figures/{image_path.stem}": wandb.Image(
                            str(image_path)
                        )
                    }
                )

            artifact = wandb.Artifact(
                f"yolo26n-fpnpan-evaluation-{CFG['mode']}-seed-{seed}",
                type="evaluation",
                metadata={
                    "architecture": "FPN-PAN",
                    "mode": CFG["mode"],
                    "seed": seed,
                    "dataset_fingerprint": DATASET_FINGERPRINT,
                },
            )
            artifact.add_dir(str(item["eval_dir"]))
            logged = run.log_artifact(
                artifact,
                aliases=["latest", f"seed-{seed}"],
            )
            logged.wait()

            WANDB_EVAL_UPLOAD_STATUS.append(
                {"seed": seed, "status": "uploaded", "run_url": run.url}
            )

        except Exception as exc:
            WANDB_EVAL_UPLOAD_STATUS.append(
                {
                    "seed": seed,
                    "status": "failed",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(
                f"[WARN] Upload evaluasi seed {seed} gagal: "
                f"{type(exc).__name__}: {exc}"
            )

        finally:
            if run is not None:
                run.finish()
else:
    print("W&B nonaktif; hasil evaluasi tetap tersedia lokal.")

(REPORTS_ROOT / "wandb_evaluation_upload_status.json").write_text(
    json.dumps(WANDB_EVAL_UPLOAD_STATUS, indent=2),
    encoding="utf-8",
)


## S. Packaging artefak

Bundle smoke dan research memakai nama berbeda. Checkpoint, audit, evaluation, metadata, tabel, dan checksum digabung menjadi satu ZIP.


In [ ]:
FINAL_DIR = WORK_ROOT / "final_artifacts"
FINAL_DIR.mkdir(exist_ok=True)

bundle_name = f"yolo26n_baseline_{CFG['mode']}_bundle"
bundle_root = FINAL_DIR / bundle_name

if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir()

(bundle_root / "config.json").write_text(
    json.dumps(CFG, indent=2),
    encoding="utf-8",
)
(bundle_root / "environment.json").write_text(
    json.dumps(ENV_INFO, indent=2),
    encoding="utf-8",
)
(bundle_root / "dataset_manifest.json").write_text(
    json.dumps(
        {
            "workspace": CFG["workspace"],
            "project": CFG["project"],
            "version": CFG["dataset_version"],
            "fingerprint": DATASET_FINGERPRINT,
            "classes": names,
            "audit_summary": summary.to_dict(orient="records"),
            "mode": CFG["mode"],
        },
        indent=2,
    ),
    encoding="utf-8",
)
(bundle_root / "RECOVERY.md").write_text(
    "# Recovery\n\n"
    "Gunakan `last.pt` untuk resume training yang terputus dan `best.pt` "
    "untuk evaluasi/inferensi. Verifikasi fingerprint dataset sebelum resume.\n",
    encoding="utf-8",
)


def safe_copy(src, dst):
    src = Path(src)
    if not src.exists():
        return

    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


for report in REPORTS_ROOT.rglob("*"):
    if report.is_file():
        safe_copy(
            report,
            bundle_root / "reports" / report.relative_to(REPORTS_ROOT),
        )

for seed, run_dir in TRAIN_DIRS.items():
    seed_root = bundle_root / f"seed_{seed}"

    for filename in ("best.pt", "last.pt"):
        safe_copy(
            run_dir / "weights" / filename,
            seed_root / "weights" / filename,
        )

    safe_copy(
        run_dir / "run_metadata.json",
        seed_root / "run_metadata.json",
    )

    eval_dir = run_dir / "evaluation"
    if eval_dir.exists():
        shutil.copytree(
            eval_dir,
            seed_root / "evaluation",
            dirs_exist_ok=True,
        )

    for pattern in ("results.csv", "args.yaml", "results.png", "*.png"):
        for src in run_dir.glob(pattern):
            safe_copy(src, seed_root / "training" / src.name)

manifest = []
for path in sorted(path for path in bundle_root.rglob("*") if path.is_file()):
    manifest.append(
        {
            "path": str(path.relative_to(bundle_root)),
            "size": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
    )

(bundle_root / "MANIFEST_SHA256.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

zip_path = FINAL_DIR / f"{bundle_name}.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        path for path in bundle_root.rglob("*") if path.is_file()
    ):
        archive.write(path, path.relative_to(bundle_root.parent))

zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
(FINAL_DIR / f"{zip_path.name}.sha256").write_text(
    f"{zip_sha256}  {zip_path.name}\n",
    encoding="utf-8",
)

upload_status = {
    "status": "not_attempted",
    "zip": str(zip_path),
    "sha256": zip_sha256,
}

if WANDB_ENABLED:
    final_run = None
    try:
        final_run = wandb.init(
            project=CFG["wandb_project"],
            entity=CFG["wandb_entity"],
            name=f"yolo26n-{CFG['mode']}-final-bundle",
            job_type="artifact-packaging",
            reinit="finish_previous",
        )
        artifact = wandb.Artifact(
            f"yolo26n-baseline-{CFG['mode']}-complete",
            type="experiment-bundle",
            metadata={
                "mode": CFG["mode"],
                "dataset_fingerprint": DATASET_FINGERPRINT,
                "completed_seeds": list(map(int, SEED_SUMMARY.seed)),
            },
        )
        artifact.add_file(str(zip_path))
        logged = final_run.log_artifact(artifact)
        logged.wait()
        upload_status.update(
            {"status": "uploaded", "wandb_run_url": final_run.url}
        )
    except Exception as exc:
        upload_status.update(
            {
                "status": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )
    finally:
        if final_run is not None:
            final_run.finish()

(FINAL_DIR / "upload_status.json").write_text(
    json.dumps(upload_status, indent=2),
    encoding="utf-8",
)

print("ZIP final     :", zip_path)
print("SHA256        :", zip_sha256)
print("Status upload :", upload_status)


## T. Final gate — end-to-end verification

Jika cell ini lolos, smoke pipeline telah melewati download → normalize → audit → 1 epoch training → checkpoint safe point → validation threshold → test evaluation → COCO scale → efficiency → qualitative diagnostics → summary → packaging.


In [ ]:
required_reports = [
    REPORTS_ROOT / ("smoke_results.csv" if IS_SMOKE else "seed_results.csv"),
    REPORTS_ROOT / "model_architecture.csv",
    REPORTS_ROOT / "model_profile_summary.json",
]

for seed, run_dir in TRAIN_DIRS.items():
    required_paths = [
        run_dir / "weights" / "best.pt",
        run_dir / "weights" / "last.pt",
        run_dir / "evaluation" / "ultralytics_overall.json",
        run_dir / "evaluation" / "test_per_class_ap.csv",
        run_dir / "evaluation" / "coco_scale_metrics.json",
        run_dir / "evaluation" / "efficiency.json",
    ]

    missing = [str(path) for path in required_paths if not path.exists()]
    if missing:
        raise RuntimeError(
            "Pipeline belum lengkap. File berikut tidak ditemukan:\n"
            + "\n".join(missing)
        )

for path in required_reports:
    if not path.exists():
        raise RuntimeError(f"Report wajib tidak ditemukan: {path}")

if not zip_path.exists():
    raise RuntimeError("ZIP final belum terbentuk.")

print("=" * 72)
print("END-TO-END PIPELINE PASSED")
print("Mode :", CFG["mode"])
print("Seeds:", list(TRAIN_DIRS))
print("ZIP  :", zip_path)
print("=" * 72)

if IS_SMOKE:
    print(
        "Smoke test berhasil. Untuk run penelitian, ubah CFG['mode'] "
        "menjadi 'research', pilih active_seed, lalu Save & Run All pada version baru."
    )
